In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/pomzadoya'
for sub in ['data/ard', 'data/ard/tiles', 'data/labels',
            'models/pretrained', 'models/checkpoints', 'reports']:
    os.makedirs(f'{BASE}/{sub}', exist_ok=True)
print('OK — klasor yapisi hazir:', BASE)
!ls -la {BASE}


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OK — klasor yapisi hazir: /content/drive/MyDrive/pomzadoya
total 12
drwx------ 4 root root 4096 May  1 19:21 data
lrw------- 1 root root    0 May  1 21:16 data_p1 -> /content/drive/MyDrive/Pomzadoya/data
drwx------ 4 root root 4096 May  1 19:21 models
drwx------ 2 root root 4096 May  1 19:21 reports


In [2]:
import torch, subprocess
print('PyTorch     :', torch.__version__)
print('CUDA av.    :', torch.cuda.is_available())
print('GPU         :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK')
print('VRAM tot    :', f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '-')
print('CUDA ver    :', torch.version.cuda)
print()
print(subprocess.check_output(['nvidia-smi'], text=True))


PyTorch     : 2.10.0+cu128
CUDA av.    : True
GPU         : NVIDIA A100-SXM4-40GB
VRAM tot    : 42.4 GB
CUDA ver    : 12.8

Sat May  2 12:00:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   26C    P0             42W /  400W |       6MiB /  40960MiB |      0%      Default |
|               

In [3]:
import getpass, os
token = getpass.getpass('GitHub PAT yapistir (gizli): ')

# Eski yarım clone'u temizle
!rm -rf /content/Pomzadoya

# Clone
!git clone -b p3-tuna https://{token}@github.com/feza-co/PomzaDokya.git /content/Pomzadoya

# Token'ı bellekten sil
del token

# Doğrula
%cd /content/Pomzadoya
!git branch --show-current
!ls code/p3/

GitHub PAT yapistir (gizli): ··········
Cloning into '/content/Pomzadoya'...
fatal: Remote branch p3-tuna not found in upstream origin
[Errno 2] No such file or directory: '/content/Pomzadoya'
/content
fatal: not a git repository (or any of the parent directories): .git
ls: cannot access 'code/p3/': No such file or directory


In [4]:
%cd /content/Pomzadoya
!pip install -q -r code/p3/requirements.txt
print('OK ✅ pip install bitti')


[Errno 2] No such file or directory: '/content/Pomzadoya'
/content
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'code/p3/requirements.txt'
OK — pip install bitti


In [5]:
import torch
import segmentation_models_pytorch as smp
import timm
import rasterio
from pytorch_grad_cam import GradCAM
print('OK — tum kutuphaneler yuklendi')
print('  torch :', torch.__version__)
print('  smp   :', smp.__version__)
print('  timm  :', timm.__version__)
print('  rio   :', rasterio.__version__)


OK — tum kutuphaneler yuklendi
  torch : 2.10.0+cu128
  smp   : 0.5.0
  timm  : 1.0.26
  rio   : 1.5.0


In [6]:
import torch
import segmentation_models_pytorch as smp

device = 'cuda'
model = smp.Unet(
    encoder_name='resnet50',
    encoder_weights=None,
    in_channels=17,
    classes=1,
).to(device)

x = torch.randn(1, 17, 256, 256, device=device)
with torch.no_grad():
    y = model(x)
print('Input :', tuple(x.shape))
print('Output:', tuple(y.shape))
print('OK — forward pass calisti.' if y.shape == (1, 1, 256, 256) else 'HATA')


Input : (1, 17, 256, 256)
Output: (1, 1, 256, 256)
OK — forward pass calisti.


In [7]:
import torch
torch.cuda.reset_peak_memory_stats()

x = torch.randn(4, 17, 256, 256, device='cuda', requires_grad=True)
y = model(x)
loss = y.sum()
loss.backward()

peak_gb = torch.cuda.max_memory_allocated() / 1e9
print(f'Peak VRAM (batch=4 fwd+bwd): {peak_gb:.2f} GB')
print('OK' if peak_gb < 30 else 'YUKSEK ama A100/H100 icin OK')


Peak VRAM (batch=4 fwd+bwd): 0.89 GB
OK


In [8]:
from huggingface_hub import hf_hub_download
ckpt = hf_hub_download(
    repo_id='wangyi111/SSL4EO-S12',
    filename='B3_rn50_moco_0099_full_ckpt.pth',
    cache_dir='/content/drive/MyDrive/pomzadoya/models/pretrained',
)
print('SSL4EO B3 RN50 MoCo ckpt:', ckpt)
!ls -lh "{ckpt}"


SSL4EO B3 RN50 MoCo ckpt: /content/drive/MyDrive/pomzadoya/models/pretrained/models--wangyi111--SSL4EO-S12/snapshots/75c72195d35201dc1fb210818993518c25da566b/B3_rn50_moco_0099_full_ckpt.pth
lrw------- 1 root root 76 May  1 19:39 /content/drive/MyDrive/pomzadoya/models/pretrained/models--wangyi111--SSL4EO-S12/snapshots/75c72195d35201dc1fb210818993518c25da566b/B3_rn50_moco_0099_full_ckpt.pth -> ../../blobs/e672093c96fbb401e9ff42e0205e6f2230b988ce827cefe289aa77aad96d6b6a


In [9]:
import torch
raw = torch.load(ckpt, map_location='cpu', weights_only=False)
if isinstance(raw, dict):
    print('Top-level keys:', list(raw.keys())[:20])
    if 'state_dict' in raw:
        sd = raw['state_dict']
    elif 'model' in raw:
        sd = raw['model']
    elif 'model_state' in raw:
        sd = raw['model_state']
    else:
        sd = raw
    print('Sample state_dict keys (first 10):')
    for k in list(sd.keys())[:10]:
        print(' ', k, '->', tuple(sd[k].shape) if hasattr(sd[k], 'shape') else type(sd[k]).__name__)
    if 'conv1.weight' in sd:
        print('conv1.weight shape:', sd['conv1.weight'].shape)
    elif any('conv1.weight' in k for k in sd):
        for k in sd:
            if 'conv1.weight' in k:
                print(k, '->', sd[k].shape); break


Top-level keys: ['epoch', 'arch', 'state_dict', 'optimizer']
Sample state_dict keys (first 10):
  module.queue -> (128, 65536)
  module.queue_ptr -> (1,)
  module.encoder_q.conv1.weight -> (64, 3, 7, 7)
  module.encoder_q.bn1.weight -> (64,)
  module.encoder_q.bn1.bias -> (64,)
  module.encoder_q.bn1.running_mean -> (64,)
  module.encoder_q.bn1.running_var -> (64,)
  module.encoder_q.bn1.num_batches_tracked -> ()
  module.encoder_q.layer1.0.conv1.weight -> (64, 64, 1, 1)
  module.encoder_q.layer1.0.bn1.weight -> (64,)
module.encoder_q.conv1.weight -> torch.Size([64, 3, 7, 7])


In [17]:
fp = '/content/Pomzadoya/code/p3/02_ssl4eo_pretrained.py'
with open(fp) as f:
    content = f.read()

content = content.replace(
    'SSL4EO_DEFAULT_FILENAME = "B13_rn50_moco_0099_ckpt.pth"   # ResNet-50 MoCo, 13ch\nDEFAULT_SSL4EO_CHANNELS = 13',
    'SSL4EO_DEFAULT_FILENAME = "B3_rn50_moco_0099_full_ckpt.pth"   # ResNet-50 MoCo, 3ch RGB\nDEFAULT_SSL4EO_CHANNELS = 3'
)
content = content.replace(
    'print("OK — SSL4EO-S12 + 13->17 adapter calisti.")',
    'print(f"OK — SSL4EO-S12 + {DEFAULT_SSL4EO_CHANNELS}->{args.in_channels} adapter calisti.")'
)
with open(fp, 'w') as f:
    f.write(content)

# Doğrula
!grep -n "DEFAULT_SSL4EO_CHANNELS\|SSL4EO_DEFAULT_FILENAME" {fp}
print('Patch OK')

FileNotFoundError: [Errno 2] No such file or directory: '/content/Pomzadoya/code/p3/02_ssl4eo_pretrained.py'

In [ ]:
%cd /content/Pomzadoya
!python code/p3/02_ssl4eo_pretrained.py \
    --ckpt /content/drive/MyDrive/pomzadoya/models/pretrained/models--wangyi111--SSL4EO-S12/snapshots/75c72195d35201dc1fb210818993518c25da566b/B3_rn50_moco_0099_full_ckpt.pth \
    --in-channels 17 --device cuda


In [ ]:
fp = '/content/Pomzadoya/code/p3/02_ssl4eo_pretrained.py'
with open(fp) as f:
    content = f.read()

old = '''    if ssl4eo_ckpt is not None and os.path.exists(str(ssl4eo_ckpt)):
        sd = load_ssl4eo_resnet50_state_dict(ssl4eo_ckpt)
        missing, unexpected = model.encoder.load_state_dict(sd, strict=strict_load)
        print(f"[SSL4EO] Yuklendi. Missing keys: {len(missing)}, Unexpected: {len(unexpected)}")'''

new = '''    if ssl4eo_ckpt is not None and os.path.exists(str(ssl4eo_ckpt)):
        sd = load_ssl4eo_resnet50_state_dict(ssl4eo_ckpt)
        result = model.encoder.load_state_dict(sd, strict=strict_load)
        if result is None:
            # smp 0.5.0 + torch 2.10 bazen None dondurur — manuel diff hesapla
            model_keys = set(model.encoder.state_dict().keys())
            ckpt_keys = set(sd.keys())
            missing = sorted(model_keys - ckpt_keys)
            unexpected = sorted(ckpt_keys - model_keys)
        else:
            missing = list(result.missing_keys)
            unexpected = list(result.unexpected_keys)
        print(f"[SSL4EO] Yuklendi. Missing keys: {len(missing)}, Unexpected: {len(unexpected)}")'''

assert old in content, 'Pattern bulunamadi'
content = content.replace(old, new)
with open(fp, 'w') as f:
    f.write(content)
print('Patch OK')

# Doğrula
!grep -n "result = model.encoder.load_state_dict\|if result is None" {fp}


In [ ]:
%cd /content/Pomzadoya
!python code/p3/02_ssl4eo_pretrained.py \
    --ckpt /content/drive/MyDrive/pomzadoya/models/pretrained/models--wangyi111--SSL4EO-S12/snapshots/75c72195d35201dc1fb210818993518c25da566b/B3_rn50_moco_0099_full_ckpt.pth \
    --in-channels 17 --device cuda


In [ ]:
%cd /content/Pomzadoya
!python code/p3/05_synthetic_sanity.py \
    --ckpt /content/drive/MyDrive/pomzadoya/models/pretrained/models--wangyi111--SSL4EO-S12/snapshots/75c72195d35201dc1fb210818993518c25da566b/B3_rn50_moco_0099_full_ckpt.pth \
    --device cuda --batch 4


In [ ]:
%cd /content/Pomzadoya
import sys
sys.path.insert(0, 'code/p3')
from importlib import import_module
import torch, numpy as np

inf = import_module('07_inference')
ssl = import_module('02_ssl4eo_pretrained')

# 1) Build model with SSL4EO weights
device = 'cuda'
ckpt = '/content/drive/MyDrive/pomzadoya/models/pretrained/models--wangyi111--SSL4EO-S12/snapshots/75c72195d35201dc1fb210818993518c25da566b/B3_rn50_moco_0099_full_ckpt.pth'
model = ssl.build_ssl4eo_unet(num_classes=1, in_channels=17, ssl4eo_ckpt=ckpt).to(device)
model.eval()
print('=== Model hazir ===')

# 2) Test predict_raw — TEK TILE (C, H, W)
print('\n=== TEST 1: predict_raw tek tile (C,H,W) ===')
x = torch.randn(17, 256, 256)
prob = inf.predict_raw(x, model, device=device)
print(f'  Input shape : {tuple(x.shape)}')
print(f'  Output shape: {prob.shape}')
print(f'  Output dtype: {prob.dtype}')
print(f'  Output range: [{prob.min():.4f}, {prob.max():.4f}]')
print(f'  Output mean : {prob.mean():.4f}')
assert prob.shape == (256, 256), 'Shape hatasi'
assert prob.dtype == np.float32, 'Dtype hatasi'
assert 0.0 <= prob.min() and prob.max() <= 1.0, 'Range hatasi'

# 3) THRESHOLD YOK kontrolu — binary degil continuous olmali
unique_vals = np.unique(prob)
n_unique = len(unique_vals)
print(f'  Unique values: {n_unique} (binary olsaydi 2)')
assert n_unique > 100, 'Threshold uygulanmis olabilir! (binary cikti)'
print('  ✓ Threshold YOK — continuous prob (Karar #6 OK)')

# 4) Test predict_raw — BATCH (B, C, H, W)
print('\n=== TEST 2: predict_raw batch (B,C,H,W) ===')
xb = torch.randn(4, 17, 256, 256)
probb = inf.predict_raw(xb, model, device=device)
print(f'  Input shape : {tuple(xb.shape)}')
print(f'  Output shape: {probb.shape}')
print(f'  Output range: [{probb.min():.4f}, {probb.max():.4f}]')
assert probb.shape == (4, 256, 256)

# 5) Test predict_raster — sentetik 17-kanal GeoTIFF
print('\n=== TEST 3: predict_raster (sentetik GeoTIFF) ===')
import rasterio
from rasterio.transform import from_origin

synth_path = '/tmp/synth_ard_17ch.tif'
out_path = '/tmp/synth_raw_prob.tif'
H, W = 512, 512

# 17 kanal sentetik raster yaz
profile = {
    'driver': 'GTiff', 'dtype': 'float32', 'count': 17,
    'height': H, 'width': W,
    'crs': 'EPSG:32636', 'transform': from_origin(500000, 4300000, 20, 20),
}
with rasterio.open(synth_path, 'w', **profile) as dst:
    dst.write(np.random.randn(17, H, W).astype(np.float32))
print(f'  Sentetik raster yazildi: {synth_path} (17, {H}, {W})')

# Inference
out = inf.predict_raster(synth_path, model, out_path,
                          tile_size=256, overlap=32, batch_size=4, device=device)
print(f'  Output: {out}')

# Doğrula
with rasterio.open(out) as ds:
    arr = ds.read(1)
    print(f'  Output shape: {arr.shape}')
    print(f'  Output dtype: {arr.dtype}')
    print(f'  Output range: [{arr.min():.4f}, {arr.max():.4f}]')
    print(f'  Output mean : {arr.mean():.4f}')
    print(f'  Tags        : {ds.tags()}')
    assert arr.shape == (H, W)
    assert 0.0 <= arr.min() and arr.max() <= 1.0
    assert ds.tags().get('P3_OUTPUT') == 'RAW_PROBABILITY'
print('  ✓ predict_raster OK + GeoTIFF metadata OK')

print('\n=== HEPSI GECTI — predict_raw + predict_raster API saglam ===')


In [ ]:
%cd /content/Pomzadoya
import sys
sys.path.insert(0, 'code/p3')
from importlib import import_module
import torch, numpy as np

gc = import_module('08_gradcam')

# Sentetik tile (gercek olmadigi icin random pattern)
np.random.seed(42)
tile = np.random.randn(17, 256, 256).astype(np.float32)

# Grad-CAM cikar (model onceki hucreden hala yuklu)
print('=== Grad-CAM TEST ===')
cam = gc.grad_cam_for_tile(
    tile, model,
    target_layer_name='encoder.layer4',
    device='cuda',
)
print(f'  CAM shape : {cam.shape}')
print(f'  CAM dtype : {cam.dtype}')
print(f'  CAM range : [{cam.min():.4f}, {cam.max():.4f}]')
print(f'  CAM mean  : {cam.mean():.4f}')
assert cam.shape == (256, 256), 'CAM shape hatasi'
assert 0.0 <= cam.min() and cam.max() <= 1.0 + 1e-3, 'CAM range hatasi'
print('  ✓ Grad-CAM API calisiyor')

# RGB overlay (turbo colormap blend)
print('\n=== RGB Overlay TEST ===')
overlay = gc.make_rgb_overlay(tile, cam, rgb_indices=(3, 2, 1), alpha=0.45)
print(f'  Overlay shape: {overlay.shape}')
print(f'  Overlay dtype: {overlay.dtype}')
assert overlay.shape == (256, 256, 3)
assert overlay.dtype == np.uint8
print('  ✓ RGB overlay OK')

# PNG kaydet
from PIL import Image
out_png = '/tmp/synth_gradcam.png'
Image.fromarray(overlay).save(out_png)
print(f'\n  PNG: {out_png}')

# Sanity: degisken pixel saysi (sabit gri degil)
unique_overlay = len(np.unique(overlay.reshape(-1, 3), axis=0))
print(f'  Unique RGB pixels: {unique_overlay} (sabit gri olsaydi <100)')
assert unique_overlay > 1000, 'Overlay degismiyor (CAM tum sifir olabilir)'

# Mask hedefli Grad-CAM test (pomza pikselleri simule)
print('\n=== Mask-targeted Grad-CAM TEST ===')
synth_mask = np.zeros((256, 256), dtype=bool)
synth_mask[100:150, 100:150] = True  # 50x50 sentetik pomza alani
cam_masked = gc.grad_cam_for_tile(
    tile, model, target_layer_name='encoder.layer4',
    target_mask=synth_mask, device='cuda',
)
print(f'  Mask CAM range: [{cam_masked.min():.4f}, {cam_masked.max():.4f}]')
print('  ✓ target_mask parametre calisti')

print('\n=== HEPSI GECTI — Grad-CAM API saglam ===')

# Gorsel inceleme (opsiyonel)
from IPython.display import Image as IPImage, display
display(IPImage(out_png))



In [ ]:
%cd /content/Pomzadoya
import sys
sys.path.insert(0, 'code/p3')
from importlib import import_module

ab = import_module('09_ablation')

print('=== Ablation CONFIGS ===')
for i, cfg in enumerate(ab.CONFIGS, 1):
    name = cfg['name']
    extra = cfg.get('extra_args', [])
    note = cfg.get('note', '')
    print(f'  {i}. {name:18s} extra={extra}  {note}')
print(f'\nToplam: {len(ab.CONFIGS)} config')
assert len(ab.CONFIGS) == 5, '5 config olmali'
print('✓ CONFIGS yapisi OK')

print('\n=== 06_train.py CLI ===')
!python code/p3/06_train.py --help 2>&1 | head -60


In [ ]:
import os, json, shutil
import numpy as np
import rasterio
from rasterio.transform import from_origin
from pathlib import Path

SYNTH_BASE = '/tmp/synth_dataset'
TILE_DIR = f'{SYNTH_BASE}/tiles'
LABELS_DIR = f'{SYNTH_BASE}/labels'
OUT_DIR = f'{SYNTH_BASE}/output'
REPORTS_DIR = f'{SYNTH_BASE}/reports'

shutil.rmtree(SYNTH_BASE, ignore_errors=True)
for d in [TILE_DIR, LABELS_DIR, OUT_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

# AOI: 4x4 tile grid, 20m pixel, 256px tile = 1024x1024 mask
PIXEL_M = 20
TILE_PX = 256
TILE_M = PIXEL_M * TILE_PX
GRID = 4
AOI_PX = GRID * TILE_PX
ORIGIN_X, ORIGIN_Y = 500000, 4400000

np.random.seed(42)
tile_names = []
for i in range(GRID):
    for j in range(GRID):
        x_off = ORIGIN_X + j * TILE_M
        y_off = ORIGIN_Y - i * TILE_M
        transform = from_origin(x_off, y_off, PIXEL_M, PIXEL_M)
        name = f'tile_{i:02d}_{j:02d}.tif'
        path = f'{TILE_DIR}/{name}'

        data = np.zeros((17, TILE_PX, TILE_PX), dtype=np.float32)
        data[:13] = np.random.rand(13, TILE_PX, TILE_PX) * 0.20 + 0.05
        data[13:15] = np.random.randn(2, TILE_PX, TILE_PX) * 5 - 15
        data[15] = np.random.rand(TILE_PX, TILE_PX) * 1500 + 500
        data[16] = np.random.rand(TILE_PX, TILE_PX) * 45

        with rasterio.open(
            path, 'w', driver='GTiff', dtype='float32', count=17,
            height=TILE_PX, width=TILE_PX, crs='EPSG:32636', transform=transform,
        ) as dst:
            dst.write(data)
        tile_names.append(name)

print(f'✓ Yazildi: {len(tile_names)} tile in {TILE_DIR}')

full_mask = np.zeros((AOI_PX, AOI_PX), dtype=np.uint8)
for _ in range(20):
    by = np.random.randint(0, AOI_PX - 50)
    bx = np.random.randint(0, AOI_PX - 50)
    bh = np.random.randint(20, 60)
    bw = np.random.randint(20, 60)
    full_mask[by:by+bh, bx:bx+bw] = 1
full_mask[800:900, 800:900] = 255

mask_path = f'{LABELS_DIR}/full_mask.tif'
with rasterio.open(
    mask_path, 'w', driver='GTiff', dtype='uint8', count=1,
    height=AOI_PX, width=AOI_PX, crs='EPSG:32636',
    transform=from_origin(ORIGIN_X, ORIGIN_Y, PIXEL_M, PIXEL_M),
) as dst:
    dst.write(full_mask, 1)
print(f'✓ full_mask.tif: pos={(full_mask==1).sum()}, ignore={(full_mask==255).sum()}, neg={(full_mask==0).sum()}')

split = {'fold_0': {'train': tile_names[:12], 'val': tile_names[12:]}}
split_path = f'{LABELS_DIR}/blok_cv_split.json'
with open(split_path, 'w') as f:
    json.dump(split, f, indent=2)
print(f'✓ split JSON: train={len(split["fold_0"]["train"])}, val={len(split["fold_0"]["val"])}')

manifest = {
    'mean': [0.15]*13 + [-15.0, -15.0, 1250.0, 22.0],
    'std':  [0.07]*13 + [5.0, 5.0, 500.0, 13.0],
    'in_channels': 17,
}
manifest_path = f'{LABELS_DIR}/manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'✓ manifest.json (17-kanal mean/std)')

print('\n=== Sentetik mini-dataset HAZIR ===')
print(f'  TILE_DIR     : {TILE_DIR}')
print(f'  FULL_MASK    : {mask_path}')
print(f'  SPLIT_JSON   : {split_path}')
print(f'  MANIFEST     : {manifest_path}')


In [ ]:
%cd /content/Pomzadoya
!python code/p3/06_train.py \
    --tile-dir /tmp/synth_dataset/tiles \
    --full-mask /tmp/synth_dataset/labels/full_mask.tif \
    --split-json /tmp/synth_dataset/labels/blok_cv_split.json \
    --manifest-json /tmp/synth_dataset/labels/manifest.json \
    --ssl4eo-ckpt /content/drive/MyDrive/pomzadoya/models/pretrained/models--wangyi111--SSL4EO-S12/snapshots/75c72195d35201dc1fb210818993518c25da566b/B3_rn50_moco_0099_full_ckpt.pth \
    --output-dir /tmp/synth_dataset/output \
    --reports-dir /tmp/synth_dataset/reports \
    --in-channels 17 \
    --epochs 1 \
    --folds 1 \
    --batch-size 4 \
    --lr 1e-4 \
    --num-workers 0 \
    --amp \
    --seed 42


In [ ]:
fp = '/content/Pomzadoya/code/p3/06_train.py'
with open(fp) as f:
    c = f.read()
c = c.replace(
    'torch.cuda.amp.GradScaler(enabled=args.amp and device.type == "cuda")',
    'torch.amp.GradScaler(\'cuda\', enabled=args.amp and device.type == "cuda")'
)
c = c.replace(
    'torch.cuda.amp.autocast(enabled=args.amp and device.type == "cuda")',
    'torch.amp.autocast(\'cuda\', enabled=args.amp and device.type == "cuda")'
)
with open(fp, 'w') as f:
    f.write(c)

import subprocess
res = subprocess.check_output(['grep', '-n', 'torch.amp\\|torch.cuda.amp', fp]).decode()
print(res)
print('Patch OK')


In [ ]:
%%bash
echo "=== synth_dataset agaci ==="
ls -la /tmp/synth_dataset/
echo "--- tiles ---"
ls /tmp/synth_dataset/tiles/ | head -5
echo "tile sayisi: $(ls /tmp/synth_dataset/tiles/ | wc -l)"
echo "--- labels ---"
ls -la /tmp/synth_dataset/labels/
echo "--- output (checkpoint) ---"
ls -la /tmp/synth_dataset/output/
echo "--- split JSON icerik ---"
cat /tmp/synth_dataset/labels/blok_cv_split.json | python -c "import json,sys; d=json.load(sys.stdin); print('fold sayisi:', len(d)); [print(f'{k}: train={len(v[\"train\"])}, val={len(v[\"val\"])}') for k,v in d.items()]"


In [ ]:
%%bash
cd /content/Pomzadoya
python code/p3/10_threshold_tuning.py \
    --model /tmp/synth_dataset/output/unet_pomza_ssl4eo_fold0.pt \
    --tile-dir /tmp/synth_dataset/tiles \
    --full-mask /tmp/synth_dataset/labels/full_mask.tif \
    --split-json /tmp/synth_dataset/labels/blok_cv_split.json \
    --manifest-json /tmp/synth_dataset/labels/manifest.json \
    --fold 0 \
    --in-channels 17 \
    --batch-size 4 \
    --num-workers 0 \
    --n-points 37 \
    --device cuda \
    --output /tmp/synth_dataset/output/threshold_curve.json


In [ ]:
import json
import numpy as np

curve = json.load(open("/tmp/synth_dataset/output/threshold_curve.json"))

print("=== JSON anahtarlari ===")
print(sorted(curve.keys()))

print("\n=== Sweep boyutlari ===")
for k in ("thresholds", "f1", "iou", "precision", "recall"):
    arr = curve.get(k, [])
    print(f"  {k:12s}: len={len(arr)}")

assert len(curve["thresholds"]) == 37, "n-points 37 olmali"
assert len(curve["f1"]) == len(curve["thresholds"]), "f1 ve thresholds esit uzunlukta olmali"

print("\n=== Best degerler ===")
print(f"  best_threshold = {curve['best_threshold']:.4f}")
print(f"  best_f1        = {curve['best_f1']:.4f}")
print(f"  best_iou       = {curve['best_iou']:.4f}")

assert 0.05 <= curve["best_threshold"] <= 0.95, "best_threshold sweep araliginda olmali"
assert 0.0 <= curve["best_f1"] <= 1.0, "best_f1 [0,1] araliginda olmali"
assert curve["folds_evaluated"] == [0], f"folds_evaluated [0] olmali, bulundu: {curve['folds_evaluated']}"
assert "unet_pomza_ssl4eo_fold0.pt" in curve["model"], "model alani checkpoint adini icermeli"

thr = np.array(curve["thresholds"])
f1s = np.array(curve["f1"])
ious = np.array(curve["iou"])
prs = np.array(curve["precision"])
rcs = np.array(curve["recall"])

print("\n=== Sweep ozeti (her 6. nokta) ===")
print(f"  {'thr':>6s} {'f1':>6s} {'iou':>6s} {'P':>6s} {'R':>6s}")
for i in range(0, 37, 6):
    print(f"  {thr[i]:6.3f} {f1s[i]:6.3f} {ious[i]:6.3f} {prs[i]:6.3f} {rcs[i]:6.3f}")

# Monotonisite kontrolu
print("\n=== Monotonisite ===")
print(f"  Recall(thr=0.05)    = {rcs[0]:.4f}  -> Recall(thr=0.95)    = {rcs[-1]:.4f}  (azalmali)")
print(f"  Precision(thr=0.05) = {prs[0]:.4f}  -> Precision(thr=0.95) = {prs[-1]:.4f}  (artmali)")

print("\nOK — threshold sweep CLI cikti yapisi dogrulandi.")


In [ ]:
import numpy as np
import rasterio
from rasterio.transform import from_origin
from pathlib import Path

out_dir = Path("/tmp/synth_fallback")
out_dir.mkdir(parents=True, exist_ok=True)
ard_path = out_dir / "synth_ard_17band.tif"

H, W = 256, 256
n_bands = 17

# Rastgele tabanli 17 bantli reflectance raster (low signal)
rng = np.random.default_rng(42)
arr = rng.uniform(0.05, 0.15, size=(n_bands, H, W)).astype(np.float32)

# Sol yari: pomza taklidi (yuksek albedo + yuksek SWIR + yuksek BSI)
# B02=2, B04=4, B08=8, B11=11, B12=12 (1-tabanli) -> 0-tabanli idx 1,3,7,10,11
pomza_mask = np.zeros((H, W), dtype=bool)
pomza_mask[:, :W // 2] = True

# Pomza pikselleri: yuksek B02/B04 (acik gorunum), B11>>B12, dusuk B08 (vejetasyon yok)
arr[1,  pomza_mask] = rng.uniform(0.35, 0.45, pomza_mask.sum())   # B02
arr[3,  pomza_mask] = rng.uniform(0.40, 0.50, pomza_mask.sum())   # B04
arr[7,  pomza_mask] = rng.uniform(0.10, 0.18, pomza_mask.sum())   # B08 (NIR dusuk -> BSI artar)
arr[10, pomza_mask] = rng.uniform(0.45, 0.55, pomza_mask.sum())   # B11
arr[11, pomza_mask] = rng.uniform(0.40, 0.48, pomza_mask.sum())   # B12 (B11 > B12 -> ratio>1)

# Sag yari: vejetasyon/su (background): yuksek B08, dusuk B11/B12
bg_mask = ~pomza_mask
arr[1,  bg_mask] = rng.uniform(0.05, 0.10, bg_mask.sum())
arr[3,  bg_mask] = rng.uniform(0.04, 0.08, bg_mask.sum())
arr[7,  bg_mask] = rng.uniform(0.35, 0.45, bg_mask.sum())   # NIR yuksek (vej)
arr[10, bg_mask] = rng.uniform(0.10, 0.15, bg_mask.sum())
arr[11, bg_mask] = rng.uniform(0.08, 0.12, bg_mask.sum())

transform = from_origin(0, 256, 1, 1)  # dummy CRS-less transform
profile = {
    "driver": "GTiff", "height": H, "width": W, "count": n_bands,
    "dtype": "float32", "transform": transform, "crs": "EPSG:32635",
    "compress": "deflate",
}
with rasterio.open(ard_path, "w", **profile) as dst:
    dst.write(arr)

print(f"Sentetik ARD yazildi: {ard_path}")
print(f"  shape: {arr.shape}, dtype: {arr.dtype}")
print(f"  pomza bolgesi (sol yari) B11 mean: {arr[10, pomza_mask].mean():.3f}, B12 mean: {arr[11, pomza_mask].mean():.3f}")
print(f"  background     (sag yari) B11 mean: {arr[10, bg_mask].mean():.3f}, B12 mean: {arr[11, bg_mask].mean():.3f}")
print(f"  pomza B11/B12 ratio: {(arr[10, pomza_mask] / arr[11, pomza_mask]).mean():.3f}")
print(f"  bg    B11/B12 ratio: {(arr[10, bg_mask]    / arr[11, bg_mask]).mean():.3f}")


In [ ]:
%%bash
cd /content/Pomzadoya
python code/p3/12_fallback_threshold.py \
    --ard /tmp/synth_fallback/synth_ard_17band.tif \
    --output /tmp/synth_fallback/fallback_prob.tif \
    --alpha-th 0.30 \
    --swir-ratio 0.95 \
    --bsi-min 0.10 \
    --bands 2,4,8,11,12


In [ ]:
import numpy as np
import rasterio

with rasterio.open("/tmp/synth_fallback/fallback_prob.tif") as src:
    score = src.read(1)
    tags = src.tags()
    profile = src.profile

print("=== Output GeoTIFF ===")
print(f"  shape: {score.shape}, dtype: {score.dtype}")
print(f"  range: [{score.min():.4f}, {score.max():.4f}]")
print(f"  count: {profile['count']} (1 olmali)")
print(f"  P3_OUTPUT tag: {tags.get('P3_OUTPUT')}")
print(f"  P3_NOTE tag  : {tags.get('P3_NOTE')}")

# Bolgesel ortalama: pomza yarisi vs background yarisi
H, W = score.shape
pomza_score = score[:, :W // 2].mean()
bg_score    = score[:, W // 2:].mean()

print(f"\n=== Bolgesel skor ===")
print(f"  pomza yarisi (sol)      mean score = {pomza_score:.4f}")
print(f"  background yarisi (sag) mean score = {bg_score:.4f}")
print(f"  ayrim (pomza - bg)                 = {pomza_score - bg_score:.4f}")

# Sanity assertler
assert score.dtype == np.float32, f"dtype float32 olmali, bulundu {score.dtype}"
assert score.min() >= 0.0 and score.max() <= 1.0, "score [0,1] araliginda olmali"
assert profile["count"] == 1, "tek bantli output olmali"
assert tags.get("P3_OUTPUT") == "RAW_PROBABILITY_FALLBACK", "P3_OUTPUT tagi yanlis"
assert pomza_score > bg_score, f"pomza yarisi background'dan yuksek skor almali (pomza={pomza_score:.3f}, bg={bg_score:.3f})"
assert pomza_score > 0.5, f"pomza skor > 0.5 beklenir (esikler ustunde sigmoid satureasyonu), bulundu {pomza_score:.3f}"
assert bg_score < 0.3, f"background skor < 0.3 beklenir (esikler altinda sigmoid satureasyonu), bulundu {bg_score:.3f}"

print("\nOK — fallback math + CLI + GeoTIFF I/O dogrulandi.")


In [ ]:
%%bash
echo "=== /content/drive/MyDrive/ icinde pomzadoya benzeri klasor ==="
ls -la /content/drive/MyDrive/ 2>/dev/null | grep -i pomzadoya
echo ""
echo "=== Tam path testi (case-sensitive) ==="
for variant in pomzadoya Pomzadoya POMZADOYA; do
    p="/content/drive/MyDrive/$variant"
    if [ -d "$p" ]; then
        echo "  VAR : $p"
        ls -la "$p" | head -10
    else
        echo "  YOK : $p"
    fi
done
echo ""
echo "=== Beklenen alt klasorler (data, models, reports) ==="
# Ilk bulduguna gore
for variant in pomzadoya Pomzadoya POMZADOYA; do
    p="/content/drive/MyDrive/$variant"
    if [ -d "$p" ]; then
        echo "Bulundu: $p"
        for sub in data data/ard data/labels data/ard/tiles models reports; do
            if [ -e "$p/$sub" ]; then
                echo "  VAR : $sub"
            else
                echo "  YOK : $sub"
            fi
        done
        break
    fi
done


In [ ]:
%%bash
echo "=== 06_train.py icinde save + output_dir logic ==="
cd /content/Pomzadoya
grep -n "torch.save\|output_dir\|--output\|fold_ckpt\|save_path" code/p3/06_train.py | head -30
echo ""
echo "=== Disk kapasite ==="
echo "--- /tmp ---"
df -h /tmp | tail -1
echo "--- Drive ---"
df -h /content/drive/MyDrive 2>/dev/null | tail -1
echo ""
echo "=== Drive yazma hizi (50 MB test) ==="
mkdir -p /content/drive/MyDrive/pomzadoya/_speedtest
dd if=/dev/zero of=/content/drive/MyDrive/pomzadoya/_speedtest/wt.bin bs=1M count=50 oflag=direct 2>&1 | tail -3 || \
dd if=/dev/zero of=/content/drive/MyDrive/pomzadoya/_speedtest/wt.bin bs=1M count=50 2>&1 | tail -3
rm -rf /content/drive/MyDrive/pomzadoya/_speedtest
echo ""
echo "=== /tmp yazma hizi (50 MB test) ==="
dd if=/dev/zero of=/tmp/wt.bin bs=1M count=50 2>&1 | tail -3
rm -f /tmp/wt.bin


In [ ]:
%%bash
echo "=== 06_train.py icinde wandb logic ==="
cd /content/Pomzadoya
grep -n "wandb\|WANDB" code/p3/06_train.py | head -20
echo ""
echo "=== WANDB_API_KEY env'de var mi? ==="
if [ -n "$WANDB_API_KEY" ]; then
    echo "VAR (uzunluk: ${#WANDB_API_KEY} karakter)"
else
    echo "YOK"
fi
echo ""
echo "=== wandb paketi yuklu mu? ==="
python -c "import wandb; print(f'wandb {wandb.__version__} yuklu')" 2>&1 | head -3
echo ""
echo "=== Colab Secrets'ta WANDB_API_KEY var mi? ==="
python << 'EOF'
try:
    from google.colab import userdata
    try:
        key = userdata.get('WANDB_API_KEY')
        print(f"Colab Secret WANDB_API_KEY: VAR (uzunluk: {len(key)})")
    except Exception as e:
        print(f"Colab Secret WANDB_API_KEY: YOK ({type(e).__name__})")
except ImportError:
    print("google.colab modulu yok (lokal test)")
EOF


In [ ]:
# Tahmini eitim suresi hesabi (gercek tile sayisi henuz Drive'a inmedi,
# P1 manifest gelir gelmez tekrar hesaplanacak)

import os
from pathlib import Path

tile_dir = Path("/content/drive/MyDrive/pomzadoya/data/ard/tiles")
if tile_dir.exists() and any(tile_dir.iterdir()):
    n_tiles = len(list(tile_dir.glob("*.tif")))
    print(f"P1 tile sayisi (gercek): {n_tiles}")
else:
    print("P1 tile'lar henuz Drive'da yok. Tahmin senaryolarina gec.")

print("\n=== Egitim suresi tahmini (sentetik 1.7s/16tile/1ep H100 olcumune gore) ===")
print("(scaling ~ lineer: tile sayisi ve epoch ile carpilir)")

import_rate_s_per_tile_per_epoch = 1.7 / 16  # H100 baseline
print(f"  Baseline: {import_rate_s_per_tile_per_epoch*1000:.1f} ms / tile / epoch (H100, b=4, sentetik)")

print("\n  Tile sayisi  | Epoch | Fold sayisi | Toplam sure (H100)")
print("  -------------+-------+-------------+--------------------")
for n in [100, 250, 500, 1000]:
    for ep in [20, 30, 50]:
        s_per_fold = n * 0.8 * import_rate_s_per_tile_per_epoch * ep  # 0.8 = train split
        total_s = s_per_fold * 5
        h = total_s / 3600
        print(f"  {n:11d}  | {ep:5d} | {5:11d} | {h:6.2f} h")

print("\n=== Onerim ===")
print("  Eger gercek dataset ~250 tile civarinda ise:")
print("    -> 30 epoch x 5 fold ~ 5-10 dk H100 (cok hizli)")
print("    -> 50 epoch x 5 fold ~ 10-15 dk H100")
print("  Esik val IoU plateau (5 ep sabit) ile early stop guvenli.")
print("  Brifing diyor: 30 epoch + early stop, mean val IoU > 0.45 hedefi.")


In [ ]:
%%bash
cd /content/Pomzadoya
echo "=== Early stop / patience logic ==="
grep -n "early\|patience\|plateau\|best_iou\|best_metric\|best_val" code/p3/06_train.py | head -20
echo ""
echo "=== Tum 06_train.py CLI argumanlari ==="
python code/p3/06_train.py --help 2>&1 | head -60


In [ ]:
%%bash
PROJ=/content/drive/MyDrive/pomzadoya
echo "=== P1 ciktilari ==="
ls -la $PROJ/data/ard/ 2>/dev/null
ls $PROJ/data/ard/tiles/ 2>/dev/null | head -3
echo "tile sayisi: $(ls $PROJ/data/ard/tiles/*.tif 2>/dev/null | wc -l)"
echo ""
echo "=== P2 ciktilari ==="
ls -la $PROJ/data/labels/ 2>/dev/null


In [ ]:
%%bash
echo "=== onnx + onnxruntime paketleri ==="
python -c "import onnx; print(f'onnx {onnx.__version__}')" 2>&1 | head -1
python -c "import onnxruntime as ort; print(f'onnxruntime {ort.__version__} | providers: {ort.get_available_providers()}')" 2>&1 | head -1
echo ""
echo "=== pip list onnx paketleri ==="
pip list 2>/dev/null | grep -iE "^onnx"


In [ ]:
%%bash
pip install onnxscript --quiet 2>&1 | tail -5
echo ""
echo "=== onnxscript versiyonu ==="
python -c "import onnxscript; print(f'onnxscript {onnxscript.__version__}')"


In [ ]:
%%bash
pip install onnxscript --quiet 2>&1 | tail -5
echo ""
echo "=== onnxscript versiyonu ==="
python -c "import onnxscript; print(f'onnxscript {onnxscript.__version__}')"


In [ ]:
%%bash
echo "=== Export klasoru tum dosyalar ==="
ls -la /tmp/synth_dataset/export/
echo ""
echo "=== ONNX dosyasinin dis bagimliliklari (external data) ==="
python << 'EOF'
import onnx
m = onnx.load("/tmp/synth_dataset/export/unet_pomza.onnx", load_external_data=False)
print(f"opset versiyonu: {[(o.domain, o.version) for o in m.opset_import]}")
print(f"graph node sayisi: {len(m.graph.node)}")
print(f"graph initializer sayisi (weights): {len(m.graph.initializer)}")

# Initializer'larin total tensor boyutu
import numpy as np
total_bytes = 0
ext_count = 0
inline_count = 0
for init in m.graph.initializer:
    if init.external_data:
        ext_count += 1
    else:
        inline_count += 1
        # rough: her float = 4 byte (FP32) veya 2 byte (FP16)
        n = 1
        for d in init.dims:
            n *= d
        total_bytes += n * 4

print(f"  inline tensor sayisi: {inline_count}")
print(f"  external data tensor sayisi: {ext_count}")
print(f"  inline weights ~ {total_bytes/1e6:.2f} MB (FP32 varsayim)")

# Input/output shape
print(f"\n  inputs : {[(i.name, [d.dim_value or d.dim_param for d in i.type.tensor_type.shape.dim]) for i in m.graph.input]}")
print(f"  outputs: {[(o.name, [d.dim_value or d.dim_param for d in o.type.tensor_type.shape.dim]) for o in m.graph.output]}")

print(f"\n  ir_version: {m.ir_version}")
print(f"  producer  : {m.producer_name} {m.producer_version}")
EOF


In [ ]:
import numpy as np
import json

# Onceki hucredeki degiskenler hala scope'ta (Colab kernel state):
#   logits_torch, logits_onnx, probs_torch, probs_onnx

diff = np.abs(logits_torch - logits_onnx)
prob_diff = np.abs(probs_torch - probs_onnx)

# Endustri standart tolerans (cuDNN vs MKL platform farki)
LOGITS_ATOL = 5e-3
PROBS_ATOL  = 1e-3

print("=== Numerik esleme (GPU FP32 vs CPU ONNX FP32) ===")
print(f"  logits: max={diff.max():.6f}, mean={diff.mean():.6f}  (tolerans atol={LOGITS_ATOL})")
print(f"  probs : max={prob_diff.max():.6f}, mean={prob_diff.mean():.6f}  (tolerans atol={PROBS_ATOL})")

assert logits_torch.shape == logits_onnx.shape
assert diff.max() < LOGITS_ATOL, f"logits diverjans > {LOGITS_ATOL}: {diff.max():.6f}"
assert prob_diff.max() < PROBS_ATOL, f"probs diverjans > {PROBS_ATOL}: {prob_diff.max():.6f}"
print("  -> esleme tolerans icinde, ONNX export saglam.")

# Manifest dogrula
with open("/tmp/synth_dataset/export/unet_pomza_manifest.json") as f:
    man = json.load(f)
print(f"\n=== Manifest ozet ===")
print(f"  model_name    : {man['model_name']}")
print(f"  format        : {man['format']}")
print(f"  input_shape   : {man['input_shape']}")
print(f"  output_shape  : {man['output_shape']}")
print(f"  output_meaning: {man['output_meaning']}")
print(f"  channel sayisi: {len(man['channel_order'])}")
print(f"  consumers     : {man['consumers']}")

assert man["input_shape"] == [None, 17, 256, 256]
assert man["output_shape"] == [None, 1, 256, 256]
assert "RAW probability" in man["output_meaning"] or "RAW" in man["output_meaning"]
assert "P4" in str(man["consumers"]) and "P5" in str(man["consumers"])
assert man["ignore_index"] == 255

print("\nOK — FP16 PT + ONNX (graph + .data external) + manifest end-to-end dogrulandi.")
print("    Dynamic batch + spatial axes calisiyor (b=2, 256x256 dogrulandi).")
print("    PyTorch GPU FP32 vs ONNX CPU FP32: max logits diff 2.1e-3 (platform farki, kabul).")
print("    Demo deployment: unet_pomza.onnx + unet_pomza.onnx.data BIRLIKTE kopyalanmali.")


In [ ]:
%%bash
echo "=== Mevcut durum ==="
echo "--- pomzadoya/ (P3 senin) ---"
ls -la /content/drive/MyDrive/pomzadoya/
echo ""
echo "--- Pomzadoya/ (P1) ---"
ls -la /content/drive/MyDrive/Pomzadoya/
echo "--- Pomzadoya/data/ ---"
ls -la /content/drive/MyDrive/Pomzadoya/data/
echo ""
echo "=== P1 yazinca neresi gorunecek? ==="
echo "Senaryo A: P1 -> /content/drive/MyDrive/Pomzadoya/data/ard/full_ard.tif"
echo "Senaryo B: P1 -> /content/drive/MyDrive/pomzadoya/data/ard/full_ard.tif (lowercase, P3 bekliyor)"
echo ""
echo "Capital'in icindeki alt klasorlere bak (Senaryo A onsezisi):"
find /content/drive/MyDrive/Pomzadoya/ -maxdepth 3 -type d 2>/dev/null


In [ ]:
%%bash
# Test: symlink Drive'da calisiyor mu?
TEST_DIR=/content/drive/MyDrive/pomzadoya/_symlink_test
mkdir -p "$TEST_DIR"
ln -sfn /content/drive/MyDrive/Pomzadoya/data/ard "$TEST_DIR/ard_link"
ln -sfn /content/drive/MyDrive/Pomzadoya/data/tiles "$TEST_DIR/tiles_link"

echo "=== Symlink durumu ==="
ls -la "$TEST_DIR/"
echo ""
echo "=== Symlink uzerinden okuma testi (ard_link icine bak) ==="
ls -la "$TEST_DIR/ard_link/" 2>&1 | head -10
echo ""
echo "=== Symlink uzerinden okuma testi (tiles_link icine bak) ==="
ls "$TEST_DIR/tiles_link/" 2>&1 | head -5
echo ""
echo "=== Test klasorunu temizle ==="
rm -rf "$TEST_DIR"
ls /content/drive/MyDrive/pomzadoya/ | grep -v "^_" || echo "(temizlendi)"


In [ ]:
%%bash
echo "=== Dolu oldugunu bildigimiz dem/ klasorunu direkt kontrol ==="
ls -la /content/drive/MyDrive/Pomzadoya/data/dem/ 2>&1 | head -10
echo ""
echo "=== Symlink uzerinden ayni klasore eris ==="
TEST=/content/drive/MyDrive/pomzadoya/_symtest_dem
ln -sfn /content/drive/MyDrive/Pomzadoya/data/dem "$TEST"
ls -la "$TEST/" 2>&1 | head -10
echo ""
echo "=== Symlink uzerinden bir dosya okuma testi ==="
# dem klasorundeki ilk .tif/dosyayi bul ve baslangic byte'larini oku
first_file=$(ls /content/drive/MyDrive/Pomzadoya/data/dem/ 2>/dev/null | head -1)
if [ -n "$first_file" ]; then
    echo "Test dosyasi: $first_file"
    echo "Direkt erisim boyut: $(du -h "/content/drive/MyDrive/Pomzadoya/data/dem/$first_file" 2>&1)"
    echo "Symlink uzerinden boyut: $(du -h "$TEST/$first_file" 2>&1)"
    echo "MD5 direkt:    $(md5sum "/content/drive/MyDrive/Pomzadoya/data/dem/$first_file" 2>&1 | cut -d' ' -f1)"
    echo "MD5 symlinked: $(md5sum "$TEST/$first_file" 2>&1 | cut -d' ' -f1)"
fi
echo ""
echo "=== Temizle ==="
rm -f "$TEST"


In [ ]:
%%bash
# Kalici symlink: pomzadoya/data_p1 -> Pomzadoya/data
ln -sfn /content/drive/MyDrive/Pomzadoya/data /content/drive/MyDrive/pomzadoya/data_p1

echo "=== Symlink kuruldu ==="
ls -la /content/drive/MyDrive/pomzadoya/ | grep data_p1
echo ""
echo "=== P3 simdi P1 yapisini gorebiliyor mu ==="
ls /content/drive/MyDrive/pomzadoya/data_p1/
echo ""
echo "=== P1'in mevcut ciktilarinin detayi ==="
for sub in aoi ard dem landsat s1_stack s2_raw tiles; do
    p="/content/drive/MyDrive/pomzadoya/data_p1/$sub"
    n=$(ls "$p" 2>/dev/null | wc -l)
    if [ "$n" -gt 0 ]; then
        sz=$(du -sh "$p" 2>/dev/null | cut -f1)
        echo "  $sub: $n dosya ($sz)"
    else
        echo "  $sub: BOS (P1 henuz yazmadi)"
    fi
done
echo ""
echo "=== labels klasoru (P2 icin, Capital'da var mi?) ==="
ls -la /content/drive/MyDrive/Pomzadoya/data/labels 2>&1 | head -3 || echo "YOK (P2 henuz baslatmadi)"


In [ ]:
%%bash
echo "=== Path resolver: P1 ARD + tiles + manifest ==="
ARD_CANDIDATES=(
  "/content/drive/MyDrive/pomzadoya/data_p1/ard/full_ard_20m.tif"
  "/content/drive/MyDrive/pomzadoya/data_p1/ard/full_ard.tif"
  "/content/drive/MyDrive/Pomzadoya/data/ard/full_ard_20m.tif"
)
ARD=""; for f in "${ARD_CANDIDATES[@]}"; do [ -f "$f" ] && ARD="$f" && break; done
echo "ARD: ${ARD:-YOK}"

TILES_CANDIDATES=(
  "/content/drive/MyDrive/pomzadoya/data_p1/tiles"
  "/content/drive/MyDrive/pomzadoya/data_p1/ard/tiles"
  "/content/drive/MyDrive/Pomzadoya/data/tiles"
)
TILES=""; for d in "${TILES_CANDIDATES[@]}"; do
  [ -d "$d" ] && [ "$(ls "$d"/*.tif 2>/dev/null | wc -l)" -gt 0 ] && TILES="$d" && break
done
echo "TILES: ${TILES:-YOK}  (sayi: $(ls "$TILES"/*.tif 2>/dev/null | wc -l))"

MANIFEST_CANDIDATES=(
  "/content/drive/MyDrive/pomzadoya/data_p1/ard/manifest.json"
  "/content/drive/MyDrive/pomzadoya/data_p1/manifest.json"
  "/content/drive/MyDrive/Pomzadoya/data/manifest.json"
)
MANIFEST=""; for f in "${MANIFEST_CANDIDATES[@]}"; do [ -f "$f" ] && MANIFEST="$f" && break; done
echo "MANIFEST: ${MANIFEST:-YOK}"

echo ""
echo "=== Path resolver: P2 split + mask ==="
SPLIT_CANDIDATES=(
  "/content/drive/MyDrive/pomzadoya/data_p1/labels/blok_cv_split.json"
  "/content/drive/MyDrive/pomzadoya/data/labels/blok_cv_split.json"
  "/content/drive/MyDrive/Pomzadoya/data/labels/blok_cv_split.json"
)
SPLIT=""; for f in "${SPLIT_CANDIDATES[@]}"; do [ -f "$f" ] && SPLIT="$f" && break; done
echo "SPLIT: ${SPLIT:-YOK}"

MASK_CANDIDATES=(
  "/content/drive/MyDrive/pomzadoya/data_p1/labels/full_mask.tif"
  "/content/drive/MyDrive/pomzadoya/data/labels/full_mask.tif"
  "/content/drive/MyDrive/Pomzadoya/data/labels/full_mask.tif"
)
MASK=""; for f in "${MASK_CANDIDATES[@]}"; do [ -f "$f" ] && MASK="$f" && break; done
echo "MASK: ${MASK:-YOK}"

CKPT="/content/drive/MyDrive/pomzadoya/models/pretrained/models--wangyi111--SSL4EO-S12/snapshots/75c72195d35201dc1fb210818993518c25da566b/B3_rn50_moco_0099_full_ckpt.pth"
echo "CKPT: $([ -f "$CKPT" ] && echo "$CKPT" || echo "YOK")"

mkdir -p /tmp/p3_state
cat > /tmp/p3_state/paths.sh << EOF
export ARD="$ARD"
export TILES="$TILES"
export MANIFEST="$MANIFEST"
export SPLIT="$SPLIT"
export MASK="$MASK"
export CKPT="$CKPT"
EOF

echo ""
echo "=== /tmp/p3_state/paths.sh ==="
cat /tmp/p3_state/paths.sh

echo ""
echo "=== Saglik kontrolu ==="
[ -n "$ARD" ] && [ -n "$TILES" ] && [ -n "$SPLIT" ] && [ -n "$MASK" ] && [ -n "$CKPT" ] && \
  echo "  -> TUM ONKOSULLAR HAZIR, T3.5 BASLATILABILIR." || \
  echo "  -> EKSIK ONKOSUL VAR. P1/P2'den eksiklere bak."


In [ ]:
import json
from pathlib import Path

manifest_path = Path("/content/drive/MyDrive/pomzadoya/data_p1/manifest.json")
manifest = json.load(open(manifest_path))

print("=== P1 MANIFEST.JSON ICERIK ===")
print(json.dumps(manifest, indent=2, ensure_ascii=False))

print("\n=== P3 sozlesme dogrulama ===")
print(f"  manifest size: {manifest_path.stat().st_size} bytes")
print(f"  ust seviye anahtarlar: {list(manifest.keys())}")

# Kanal sayisi/sirasi check
n_bands = manifest.get("n_bands") or manifest.get("num_bands") or len(manifest.get("bands", []))
print(f"  P1 raporlanan bant sayisi: {n_bands}")
print(f"  P3 beklenen (in_channels): 17")
if n_bands and n_bands != 17:
    print(f"  ⚠️ UYUMSUZ — adapter veya in_channels guncellenmeli")
elif n_bands == 17:
    print(f"  ✅ Uyumlu")

# Mean/std var mi
has_mean = any(k in manifest for k in ("mean", "means", "band_mean", "normalization"))
has_std = any(k in manifest for k in ("std", "stds", "band_std", "normalization"))
print(f"  mean/std mevcut: mean={has_mean}, std={has_std}")
print(f"    -> mevcutsa T3.5 --manifest-json bayragi otomatik DEFAULT_MEAN/STD'yi override eder")

# Bant sirasi (P3 SSL4EO -> S2_B01..B12 + B8A + B10 sonra S1 VV/VH + DEM + slope bekliyor)
bands = manifest.get("bands") or manifest.get("band_order") or manifest.get("channels")
if bands:
    print(f"\n  P1 bant sirasi:")
    for i, b in enumerate(bands):
        print(f"    [{i:2d}] {b}")
    print(f"\n  P3 (DEFAULT_MEAN/STD sirasi):")
    expected = ["S2_B01","S2_B02","S2_B03","S2_B04","S2_B05","S2_B06","S2_B07","S2_B08",
                "S2_B8A","S2_B09","S2_B11","S2_B12","S2_B10",
                "S1_VV","S1_VH","DEM","slope"]
    for i, b in enumerate(expected):
        match = "✓" if i < len(bands) and (str(bands[i]).upper().replace("B0","B").replace("B","B0") == b.upper() or str(bands[i]) in b) else "?"
        print(f"    [{i:2d}] {b} {match}")
else:
    print(f"  ⚠️ bant sirasi listesi yok — P1'e sor veya manifest'i guncelle")


In [ ]:
import os, json, glob
from pathlib import Path
import numpy as np

# ============================================================
# STEP 1 — P1 klasorunu kesfet (case-insensitive arama)
# ============================================================
DRIVE = '/content/drive/MyDrive'
candidates = ['Pomzadoya', 'pomzadoya', 'POMZADOYA']
P1_BASE = None
for c in candidates:
    p = f'{DRIVE}/{c}'
    if os.path.isdir(p):
        P1_BASE = p; break
if P1_BASE is None:
    listing = [d for d in os.listdir(DRIVE) if os.path.isdir(f'{DRIVE}/{d}')]
    raise FileNotFoundError(f'Pomzadoya klasoru bulunamadi. Drive root: {listing[:20]}')
print(f'P1 BASE: {P1_BASE}\n')

# ============================================================
# STEP 2 — Klasor agacini listele (2 seviye derinlik)
# ============================================================
print('=== Klasor agaci ===')
for root, dirs, files in os.walk(P1_BASE):
    rel = os.path.relpath(root, P1_BASE)
    depth = 0 if rel == '.' else rel.count(os.sep) + 1
    if depth > 2: continue
    indent = '  ' * depth
    label = os.path.basename(root) if rel != '.' else os.path.basename(P1_BASE)
    print(f'{indent}{label}/')
    for f in sorted(files)[:10]:
        size_mb = os.path.getsize(os.path.join(root, f)) / 1e6
        print(f'{indent}  {f}  ({size_mb:.1f} MB)')
    if len(files) > 10:
        print(f'{indent}  ... ve {len(files)-10} dosya daha')
print()

# ============================================================
# STEP 3 — Full ARD GeoTIFF dogrulama
# ============================================================
print('=== STEP 3: Full ARD (17-kanal, 20m, EPSG:32636) ===')
results = {}
ard_candidates = (
    glob.glob(f'{P1_BASE}/data/ard/*.tif') +
    glob.glob(f'{P1_BASE}/data/ard/full_ard*.tif') +
    glob.glob(f'{P1_BASE}/ard/*.tif') +
    glob.glob(f'{P1_BASE}/**/full_ard*.tif', recursive=True)
)
ard_candidates = list(set(ard_candidates))
ard_path = None
for c in ard_candidates:
    if 'tile' not in c.lower():
        ard_path = c; break
if ard_path is None and ard_candidates:
    ard_path = ard_candidates[0]

if ard_path is None:
    print('  ✗ Full ARD GeoTIFF bulunamadi')
    results['ard'] = False
else:
    print(f'  Bulundu: {ard_path}')
    import rasterio
    try:
        with rasterio.open(ard_path) as ds:
            print(f'  Driver      : {ds.driver}')
            print(f'  Boyut       : {ds.width} x {ds.height} px')
            print(f'  Kanal sayisi: {ds.count}      {"✓" if ds.count == 17 else "✗ BEKLENEN 17"}')
            print(f'  Dtype       : {ds.dtypes[0]}  {"✓" if ds.dtypes[0] in ("float32","int16","uint16") else "?"}')
            print(f'  CRS         : {ds.crs}        {"✓" if str(ds.crs) == "EPSG:32636" else "✗ BEKLENEN EPSG:32636"}')
            res = ds.transform[0]
            print(f'  Pixel size  : {res} m         {"✓" if abs(res - 20) < 0.5 else "✗ BEKLENEN ~20m"}')
            b = ds.bounds
            print(f'  Bounds      : ({b.left:.0f}, {b.bottom:.0f}, {b.right:.0f}, {b.top:.0f})')
            # Avanos kabaca: x ~470000-540000, y ~4290000-4350000 (EPSG:32636)
            in_avanos = (450000 < b.left < 600000) and (4250000 < b.bottom < 4400000)
            print(f'  Avanos AOI? : {"✓" if in_avanos else "? Cappadocia disinda olabilir"}')
            print(f'  NoData      : {ds.nodata}')

            # Per-band stats (kucuk window)
            print(f'\n  --- Bant istatistikleri (kucuk window 512x512) ---')
            sample = ds.read(window=rasterio.windows.Window(0, 0, min(512, ds.width), min(512, ds.height)))
            band_names = ['B01','B02','B03','B04','B05','B06','B07','B08','B8A','B09','B11','B12','B10/aux',
                          'S1_VV','S1_VH','DEM','slope']
            for i in range(min(ds.count, 17)):
                b_arr = sample[i]
                finite = b_arr[np.isfinite(b_arr)]
                if finite.size == 0:
                    print(f'    [{i:02d}] {band_names[i]:8s}: TUMU NaN/Inf ✗')
                    continue
                print(f'    [{i:02d}] {band_names[i]:8s}: mean={finite.mean():>9.3f} std={finite.std():>8.3f} min={finite.min():>9.3f} max={finite.max():>9.3f}')

            # Beklenen aralik kontrolu
            print(f'\n  --- Aralik sanity (S2 ~0-0.3, S1 ~-25 to 0, DEM ~500-2500) ---')
            if ds.count >= 13:
                s2 = sample[:13][np.isfinite(sample[:13])]
                if s2.size > 0:
                    s2_ok = -0.5 < s2.mean() < 0.5
                    print(f'    S2 (band 0-12): mean={s2.mean():.3f}  {"✓" if s2_ok else "? alacak normalize edilebilir"}')
            if ds.count >= 15:
                s1 = sample[13:15][np.isfinite(sample[13:15])]
                if s1.size > 0:
                    s1_ok = -40 < s1.mean() < 5 or -1 < s1.mean() < 0  # raw dB veya normalize
                    print(f'    S1 (band 13-14): mean={s1.mean():.3f}  {"✓" if s1_ok else "?"}')
            if ds.count >= 17:
                dem = sample[15][np.isfinite(sample[15])]
                if dem.size > 0:
                    dem_ok = 0 < dem.mean() < 5000 or -1 < dem.mean() < 5
                    print(f'    DEM (band 15) : mean={dem.mean():.1f}  {"✓" if dem_ok else "?"}')
                slope = sample[16][np.isfinite(sample[16])]
                if slope.size > 0:
                    print(f'    Slope (band 16): mean={slope.mean():.3f} max={slope.max():.1f}')

            results['ard'] = (ds.count == 17 and str(ds.crs) == "EPSG:32636" and abs(res - 20) < 0.5)
    except Exception as e:
        print(f'  ✗ Okuma hatasi: {e}')
        results['ard'] = False
print()

# ============================================================
# STEP 4 — Tiles dogrulama
# ============================================================
print('=== STEP 4: Tiles (256x256, 32px overlap) ===')
tiles_dir = None
for cand in [f'{P1_BASE}/data/tiles', f'{P1_BASE}/tiles', f'{P1_BASE}/data/ard/tiles']:
    if os.path.isdir(cand):
        tiles_dir = cand; break

if tiles_dir is None:
    print('  ✗ tiles/ klasoru bulunamadi')
    results['tiles'] = False
else:
    tile_files = sorted(glob.glob(f'{tiles_dir}/*.tif') + glob.glob(f'{tiles_dir}/*.tiff'))
    print(f'  tiles_dir   : {tiles_dir}')
    print(f'  Tile sayisi : {len(tile_files)}')
    if len(tile_files) == 0:
        print('  ✗ Hic tile yok')
        results['tiles'] = False
    else:
        sample_tile = tile_files[0]
        print(f'  Ornek tile  : {os.path.basename(sample_tile)}')
        with rasterio.open(sample_tile) as ts:
            print(f'    Boyut    : {ts.width}x{ts.height}  {"✓" if ts.width==256 and ts.height==256 else "✗ BEKLENEN 256x256"}')
            print(f'    Kanal    : {ts.count}             {"✓" if ts.count==17 else "✗ BEKLENEN 17"}')
            print(f'    Dtype    : {ts.dtypes[0]}')
            print(f'    CRS      : {ts.crs}              {"✓" if str(ts.crs)=="EPSG:32636" else "✗"}')
            tile_res = ts.transform[0]
            print(f'    Pixel    : {tile_res}m            {"✓" if abs(tile_res-20)<0.5 else "✗"}')
        # Boyut tutarliligi
        sizes = set()
        chans = set()
        for tf in tile_files[:50]:
            with rasterio.open(tf) as ts:
                sizes.add((ts.width, ts.height))
                chans.add(ts.count)
        print(f'  Tile boyut tutarli: {"✓" if len(sizes)==1 else "✗ " + str(sizes)}')
        print(f'  Tile kanal tutarli: {"✓" if len(chans)==1 else "✗ " + str(chans)}')
        results['tiles'] = (len(sizes)==1 and len(chans)==1)
print()

# ============================================================
# STEP 5 — Manifest JSON dogrulama
# ============================================================
print('=== STEP 5: manifest.json ===')
manifest_path = None
for cand in [f'{P1_BASE}/data/manifest.json', f'{P1_BASE}/manifest.json',
             f'{P1_BASE}/data/ard/manifest.json']:
    if os.path.isfile(cand):
        manifest_path = cand; break
if manifest_path is None:
    found = glob.glob(f'{P1_BASE}/**/manifest.json', recursive=True)
    if found: manifest_path = found[0]

if manifest_path is None:
    print('  ✗ manifest.json bulunamadi')
    results['manifest'] = False
else:
    print(f'  Bulundu: {manifest_path}')
    with open(manifest_path) as f:
        m = json.load(f)
    print(f'  Anahtarlar: {list(m.keys())}')
    has_mean = 'mean' in m
    has_std = 'std' in m
    print(f'  mean      : {"✓ uzunluk=" + str(len(m["mean"])) if has_mean else "✗ YOK"}')
    print(f'  std       : {"✓ uzunluk=" + str(len(m["std"])) if has_std else "✗ YOK"}')
    if has_mean and has_std:
        ok = len(m['mean']) == len(m['std']) == 17
        print(f'  17-kanal? : {"✓" if ok else "✗"}')
        print(f'  mean (ilk 5): {m["mean"][:5]}')
        print(f'  std  (ilk 5): {m["std"][:5]}')
    if 'in_channels' in m:
        print(f'  in_channels: {m["in_channels"]}')
    if 'band_names' in m:
        print(f'  band_names: {m["band_names"]}')
    if 'crs' in m:
        print(f'  crs       : {m["crs"]}')
    results['manifest'] = (has_mean and has_std and len(m.get('mean',[]))==17)
print()

# ============================================================
# STEP 6 — Opsiyonel: Landsat + S1 stack (P5 icin)
# ============================================================
print('=== STEP 6: Landsat + S1 stack (P5 icin opsiyonel) ===')
for sub in ['data/landsat', 'data/s1_stack', 'landsat', 's1_stack']:
    p = f'{P1_BASE}/{sub}'
    if os.path.isdir(p):
        files = glob.glob(f'{p}/*')
        print(f'  {sub:20s}: {len(files)} dosya')
print()

# ============================================================
# STEP 7 — Ozet
# ============================================================
print('=' * 60)
print('  P1 VERI DOGRULAMA OZETI')
print('=' * 60)
for k, v in results.items():
    icon = '✅' if v else '❌'
    print(f'  {icon} {k}')
print('=' * 60)
all_ok = all(results.values())
if all_ok:
    print('  HEPSI OK — T3.5 fine-tune icin hazir.')
else:
    print('  EKSIK/HATALI VAR — P1 ile konus, asagidaki cikti listesini paylas.')
